In [2]:
import sys, os
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

In [3]:
from phase1.test_cases import SOMA_PIECES
from hybrid_solver import hybrid_solve
print("pieces:", len(SOMA_PIECES))



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/nick/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/nick/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/nick/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  File "/Users/nick/ana

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [1]:
import sys
print(sys.executable)


/Users/nick/Desktop/Polycube Solver/.venv/bin/python


In [2]:
import sys
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")
from phase1.test_cases import SOMA_PIECES
print(len(SOMA_PIECES))


7


In [3]:
from hybrid_solver import hybrid_solve
from phase1.test_cases import SOMA_PIECES

result = hybrid_solve(SOMA_PIECES, grid_size=3, model_name=None, verbose=True)
print("method:", result["method"])
print("solved:", result["solution"] is not None)
print("time:", round(result["time"], 3), "sec")


Solving 3x3x3 cube (7 pieces, 27 cells)

[2] DLX exact cover solver...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 1 solution(s).
    DLX solved in 0.138s
method: dlx
solved: True
time: 0.138 sec


In [4]:
result = hybrid_solve(
    SOMA_PIECES,
    grid_size=3,
    model_name="soma_3x3x3_quick",
    beam_width=64,
    timeout_nn=10,
    verbose=True
)
print("method:", result["method"])
print("solved:", result["solution"] is not None)
print("time:", round(result["time"], 3), "sec")


Solving 3x3x3 cube (7 pieces, 27 cells)

[1] NN beam search (beam_width=64, timeout=10s)...
    NN solved in 3.328s!
method: nn
solved: True
time: 3.328 sec


## Change 1: policy guided search

In [1]:
# Cell 1: setup
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")
print(sys.executable)

/Users/nick/Desktop/Polycube Solver/.venv/bin/python


In [2]:
# Cell 2: reload updated modules
import phase2.nn_solver as nn_solver
import hybrid_solver as hs
importlib.reload(nn_solver)
importlib.reload(hs)
print("modules reloaded")

modules reloaded


In [3]:
# Cell 3: run twice to check determinism + behavior
from phase1.test_cases import SOMA_PIECES

r1 = hs.hybrid_solve(
    SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
    beam_width=64, timeout_nn=10, verbose=False
)
r2 = hs.hybrid_solve(
    SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
    beam_width=64, timeout_nn=10, verbose=False
)

print("run1:", r1["method"], r1["solution"] is not None, round(r1["time"], 3))
print("run2:", r2["method"], r2["solution"] is not None, round(r2["time"], 3))
print("same solution:", r1["solution"] == r2["solution"])


run1: nn True 2.477
run2: nn True 2.617
same solution: True


In [4]:
# Cell 4: optional quick mini-benchmark
times = []
methods = []
for _ in range(5):
    r = hs.hybrid_solve(
        SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
        beam_width=64, timeout_nn=10, verbose=False
    )
    times.append(r["time"])
    methods.append(r["method"])

print("methods:", methods)
print("avg time:", round(sum(times)/len(times), 3), "sec")
print("min/max:", round(min(times), 3), round(max(times), 3))


methods: ['nn', 'nn', 'nn', 'nn', 'nn']
avg time: 3.052 sec
min/max: 2.705 3.589


## Change 2: Disable Policy Loss (Temporary Safety Fix)

In [5]:
# Cell 1: reload updated training module
import sys, importlib, inspect
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.train as tr
importlib.reload(tr)

print(inspect.signature(tr.run_supervised_training))
print(inspect.signature(tr.run_adi_iteration))

(grid_size=3, max_pieces=10, num_instances=100, num_negatives=2, epochs=50, lr=0.001, batch_size=64, lambda_value=1.0, lambda_policy=0.0, hidden_dim=128, num_residual_blocks=6, device='cpu', save_name=None)
(model, grid_size=3, max_pieces=10, num_new_instances=50, beam_width=32, adi_epochs=20, lr=0.0005, batch_size=64, lambda_value=1.0, lambda_policy=0.0, existing_examples=None, device='cpu', verbose=True)


In [6]:
# Cell 2: quick smoke training (1 epoch, tiny model) with policy loss OFF
model, history, examples = tr.run_supervised_training(
    grid_size=3, max_pieces=10,
    num_instances=5, num_negatives=1,
    epochs=1, batch_size=16,
    lambda_value=1.0, lambda_policy=0.0,
    hidden_dim=32, num_residual_blocks=2,
    save_name=None, device="cpu"
)

print("train_policy_loss:", history["train_policy_loss"][-1])
print("val_policy_loss:", history["val_policy_loss"][-1])
print("train_value_loss:", history["train_value_loss"][-1])
print("val_value_loss:", history["val_value_loss"][-1])

Supervised Training Pipeline (grid=3x3x3)

[Step 1] Generating training data...
Generating Soma training data (5 instances)...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 11520 solution(s).
  Found 11520 distinct Soma solutions for data diversity
Generated 40 examples (35 positive, 5 negative)

[Step 2] Splitting data (40 examples)...
  Train: 34, Val: 6

[Step 3] Creating model...
  Parameters: 1,244,476 total, 1,244,476 trainable

[Step 4] Training for 1 epochs...
Epoch   1/1 [0.1s] train_loss=0.5338 val_loss=0.6525 val_acc=0.667 lr=0.000000

Training complete!
  Final val accuracy: 0.667
  Best val loss: 0.6525
train_policy_loss: 3.364712154164034
val_policy_loss: 3.3079864978790283
train_value_loss

In [11]:
## Change 3: Batch Policy Inference Across Beam

In [5]:
# Cell 1: reload
import sys, importlib, time
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.nn_solver as nn_solver
import hybrid_solver as hs
importlib.reload(nn_solver)
importlib.reload(hs)

from phase1.test_cases import SOMA_PIECES
print("reloaded")


reloaded


In [6]:
# Cell 2: determinism + sanity
r1 = hs.hybrid_solve(
    SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
    beam_width=64, timeout_nn=10, verbose=False
)
r2 = hs.hybrid_solve(
    SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
    beam_width=64, timeout_nn=10, verbose=False
)

print("run1:", r1["method"], r1["solution"] is not None, round(r1["time"], 3))
print("run2:", r2["method"], r2["solution"] is not None, round(r2["time"], 3))
print("same solution:", r1["solution"] == r2["solution"])


run1: nn True 3.179
run2: nn True 2.926
same solution: True


In [7]:
# Cell 3: quick timing sample
times = []
for _ in range(8):
    r = hs.hybrid_solve(
        SOMA_PIECES, grid_size=3, model_name="soma_3x3x3_quick",
        beam_width=64, timeout_nn=10, verbose=False
    )
    times.append(r["time"])

print("avg:", round(sum(times)/len(times), 3), "sec")
print("min:", round(min(times), 3), "max:", round(max(times), 3))
print("times:", [round(t, 3) for t in times])


avg: 3.309 sec
min: 2.721 max: 4.709
times: [2.774, 3.133, 2.812, 2.721, 3.279, 3.201, 4.709, 3.838]


In [13]:
## Change 4: Reduce ADI Label Noise (Verified Negatives)

In [14]:
# Cell 1: reload + signature check
import sys, importlib, inspect
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.train as tr
importlib.reload(tr)

print(inspect.signature(tr.run_adi_iteration))


(model, grid_size=3, max_pieces=10, num_new_instances=50, beam_width=32, adi_epochs=20, lr=0.0005, batch_size=64, lambda_value=1.0, lambda_policy=0.0, failed_label_mode='verify', failed_verify_fraction=0.15, failed_verify_max_states=16, existing_examples=None, device='cpu', verbose=True)


In [16]:
# Cell 2: compare failed-state labeling modes quickly (no retraining)
from phase2.train import load_model

model, _, _ = load_model("soma_3x3x3_quick", device="cpu")

print("\n--- verify mode ---")
_, _, ex_verify = tr.run_adi_iteration(
    model=model,
    grid_size=3,
    max_pieces=10,
    num_new_instances=6,
    beam_width=8,          # narrow to induce some failures
    adi_epochs=0,          # no retrain for this check
    batch_size=16,
    failed_label_mode="verify",
    failed_verify_fraction=0.25,
    failed_verify_max_states=10,
    existing_examples=None,
    device="cpu",
    verbose=True,
)

print("\n--- negative mode (legacy) ---")
_, _, ex_negative = tr.run_adi_iteration(
    model=model,
    grid_size=3,
    max_pieces=10,
    num_new_instances=6,
    beam_width=8,
    adi_epochs=0,
    batch_size=16,
    failed_label_mode="negative",
    existing_examples=None,
    device="cpu",
    verbose=True,
)

print("\nexamples verify:", len(ex_verify))
print("examples negative:", len(ex_negative))


--- verify mode ---

Autodidactic Iteration (ADI)

  ADI collection done: 76 examples (45 positive, 31 negative)
  Beam search solved 1/6 instances
  Failed-state handling: verified=50, labeled_negative=31, skipped=438

  Retraining for 0 epochs...

--- negative mode (legacy) ---

Autodidactic Iteration (ADI)

  ADI collection done: 439 examples (144 positive, 295 negative)
  Beam search solved 3/6 instances
  Failed-state handling: labeled_negative=295

  Retraining for 0 epochs...

examples verify: 76
examples negative: 439


In [17]:
# Cell 3: quick label counts
def counts(ex):
    pos = sum(1 for e in ex if e["label"] == 1.0)
    neg = sum(1 for e in ex if e["label"] == 0.0)
    return pos, neg


verify pos/neg: (45, 31)
legacy pos/neg: (144, 295)


In [18]:
## Change 5: Enforce DLX Timeout Guard

In [19]:
# Cell 1: reload
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import hybrid_solver as hs
importlib.reload(hs)

from phase1.test_cases import SOMA_PIECES
print("reloaded")


reloaded


In [20]:
# Cell 2: normal DLX fallback should solve (model_name=None forces DLX path)
r_ok = hs.hybrid_solve(
    SOMA_PIECES,
    grid_size=3,
    model_name=None,
    timeout_dlx=5,
    verbose=True
)
print("method:", r_ok["method"])
print("solved:", r_ok["solution"] is not None)
print("time:", round(r_ok["time"], 3), "sec")


Solving 3x3x3 cube (7 pieces, 27 cells)

[2] DLX exact cover solver...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 1 solution(s).
    DLX solved in 2.076s
method: dlx
solved: True
time: 2.076 sec


In [21]:
# Cell 3: forced timeout test (very tiny timeout)
r_to = hs.hybrid_solve(
    SOMA_PIECES,
    grid_size=3,
    model_name=None,
    timeout_dlx=0.001,
    verbose=True
)
print("method:", r_to["method"])
print("solved:", r_to["solution"] is not None)
print("time:", round(r_to["time"], 3), "sec")


Solving 3x3x3 cube (7 pieces, 27 cells)

[2] DLX exact cover solver...
    DLX timed out after 0.001s
method: None
solved: False
time: 0.034 sec


In [22]:
## Change 6: Grouped Train/Val Split (No Instance Leakage)

In [23]:
# Cell 1: reload + generate a small dataset
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.data_generator as dg
importlib.reload(dg)

examples = dg.generate_soma_training_data(num_instances=8, num_negatives=1, max_pieces=10)
print("examples:", len(examples))
print("has instance_id:", all("instance_id" in e for e in examples))


Generating Soma training data (8 instances)...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 11520 solution(s).
  Found 11520 distinct Soma solutions for data diversity
Generated 64 examples (56 positive, 8 negative)
examples: 64
has instance_id: True


In [24]:
# Cell 2: compare old (row-random) vs new (grouped) split leakage
tr_old, va_old = dg.split_dataset(examples, val_fraction=0.25, seed=42, group_key=None)
tr_new, va_new = dg.split_dataset(examples, val_fraction=0.25, seed=42, group_key="instance_id")

def overlap_count(train_ex, val_ex):
    tid = {e["instance_id"] for e in train_ex}
    vid = {e["instance_id"] for e in val_ex}
    return len(tid & vid), len(tid), len(vid)

old_overlap, old_tg, old_vg = overlap_count(tr_old, va_old)
new_overlap, new_tg, new_vg = overlap_count(tr_new, va_new)

print("old split overlap:", old_overlap, "| train groups:", old_tg, "| val groups:", old_vg)
print("new split overlap:", new_overlap, "| train groups:", new_tg, "| val groups:", new_vg)


old split overlap: 7 | train groups: 8 | val groups: 7
new split overlap: 0 | train groups: 6 | val groups: 2


In [25]:
# Cell 3: assertion for leakage fix
assert new_overlap == 0, "Grouped split still leaks instance IDs!"
print("PASS: grouped split has zero instance leakage")


PASS: grouped split has zero instance leakage


In [ ]:
## Change 7: Policy Target Redesign (Placement Actions)

In [26]:
# Cell 1: reload
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.data_generator as dg
import phase2.train as tr
importlib.reload(dg)
importlib.reload(tr)
print("reloaded")


reloaded


In [27]:
# Cell 2: inspect generated examples
examples = dg.generate_soma_training_data(num_instances=6, num_negatives=1, max_pieces=10)
ex_pos = next(e for e in examples if e["label"] == 1.0)

print("example keys:", sorted(ex_pos.keys()))
print("policy_candidates shape:", ex_pos["policy_candidates"].shape)
print("policy_target_idx:", ex_pos["policy_target_idx"])
print("target in range:", 0 <= ex_pos["policy_target_idx"] < ex_pos["policy_candidates"].shape[0])
# Cell 3: dataset tensors include action-space policy fields
train_ex, val_ex = dg.split_dataset(examples, val_fraction=0.25, seed=42, group_key="instance_id")
train_ds = dg.create_torch_dataset(train_ex)

sample = train_ds[0]
print("dataset keys:", sorted(sample.keys()))
print("policy_candidates tensor shape:", sample["policy_candidates"].shape)   # (max_actions, N^3)
print("policy_action_mask shape:", sample["policy_action_mask"].shape)
print("policy_target_idx:", int(sample["policy_target_idx"]))


Generating Soma training data (6 instances)...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 11520 solution(s).
  Found 11520 distinct Soma solutions for data diversity
Generated 48 examples (42 positive, 6 negative)
example keys: ['grid', 'grid_size', 'instance_id', 'label', 'next_piece_idx', 'next_placement', 'policy_candidates', 'policy_target_idx', 'remaining_pieces', 'state', 'value']
policy_candidates shape: (144, 27)
policy_target_idx: 123
target in range: True


In [28]:
# Cell 3: dataset tensors include action-space policy fields
train_ex, val_ex = dg.split_dataset(examples, val_fraction=0.25, seed=42, group_key="instance_id")
train_ds = dg.create_torch_dataset(train_ex)

sample = train_ds[0]
print("dataset keys:", sorted(sample.keys()))
print("policy_candidates tensor shape:", sample["policy_candidates"].shape)   # (max_actions, N^3)
print("policy_action_mask shape:", sample["policy_action_mask"].shape)
print("policy_target_idx:", int(sample["policy_target_idx"]))


dataset keys: ['label', 'next_piece_idx', 'next_placement', 'policy_action_mask', 'policy_candidates', 'policy_target_idx', 'state', 'value']
policy_candidates tensor shape: torch.Size([144, 27])
policy_action_mask shape: torch.Size([144])
policy_target_idx: 95


In [29]:
# Cell 4: quick train smoke with policy ON
from torch.utils.data import DataLoader

val_ds = dg.create_torch_dataset(val_ex)
model = tr.create_model(grid_size=3, max_pieces=10, num_residual_blocks=2, hidden_dim=32)

history = tr.train(
    model,
    DataLoader(train_ds, batch_size=8, shuffle=True),
    DataLoader(val_ds, batch_size=8, shuffle=False),
    epochs=1,
    lr=1e-3,
    lambda_value=1.0,
    lambda_policy=1.0,
    device="cpu",
    verbose=True,
)

print("train_loss:", history["train_loss"][-1])
print("train_value_loss:", history["train_value_loss"][-1])
print("train_policy_loss:", history["train_policy_loss"][-1])


Epoch   1/1 [0.2s] train_loss=2.7324 val_loss=3.1232 val_acc=0.875 lr=0.000000
train_loss: 2.732381522655487
train_value_loss: 0.423959456384182
train_policy_loss: 2.308422088623047


In [30]:
## Ablation: Policy Loss Off vs On (lambda_policy=0.0 vs 1.0)

In [31]:
# Cell 1: setup
import sys, random, time, importlib
import numpy as np
import torch

sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import phase2.data_generator as dg
import phase2.train as tr
import phase2.nn_solver as ns
from phase1.test_cases import SOMA_PIECES
from torch.utils.data import DataLoader

importlib.reload(dg)
importlib.reload(tr)
importlib.reload(ns)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

print("ready")


ready


In [32]:
# Cell 2: build one shared dataset (fair comparison)
set_seed(123)

NUM_INSTANCES = 40
NUM_NEG = 2
MAX_PIECES = 10

examples = dg.generate_soma_training_data(
    num_instances=NUM_INSTANCES,
    num_negatives=NUM_NEG,
    max_pieces=MAX_PIECES,
)
train_ex, val_ex = dg.split_dataset(
    examples, val_fraction=0.2, seed=42, group_key="instance_id"
)

train_ds = dg.create_torch_dataset(train_ex)
val_ds = dg.create_torch_dataset(val_ex)

print("examples:", len(examples), "train:", len(train_ds), "val:", len(val_ds))


Generating Soma training data (40 instances)...
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 144 placements
  Piece 2: 4 cubes, 72 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 4 cubes, 96 placements
  Piece 5: 4 cubes, 96 placements
  Piece 6: 4 cubes, 64 placements
  DLX matrix: 688 rows x 34 columns
Solving...
Found 11520 solution(s).
  Found 11520 distinct Soma solutions for data diversity
  Processed 10/40 instances, 90 total examples
  Processed 20/40 instances, 180 total examples
  Processed 30/40 instances, 270 total examples
  Processed 40/40 instances, 360 total examples
Generated 360 examples (280 positive, 80 negative)
examples: 360 train: 288 val: 72


In [35]:
# Cell 3: train helper
def train_variant(lambda_policy, epochs=6, seed=2026):
    set_seed(seed)
    model = tr.create_model(
        grid_size=3,
        max_pieces=MAX_PIECES,
        num_residual_blocks=4,
        hidden_dim=64,
    )

    # same shuffle seed for both variants
    g = torch.Generator()
    g.manual_seed(999)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

    history = tr.train(
        model,
        train_loader,
        val_loader,
        epochs=epochs,
        lr=1e-3,
        lambda_value=1.0,
        lambda_policy=lambda_policy,
        device="cpu",
        verbose=False,
    )
    return model, history


In [34]:
# Cell 4: train both variants
model_off, hist_off = train_variant(lambda_policy=0.0, epochs=6, seed=2026)
model_on,  hist_on  = train_variant(lambda_policy=1.0, epochs=6, seed=2026)

print("OFF val_loss:", round(hist_off["val_loss"][-1], 4),
      "| val_policy_loss:", round(hist_off["val_policy_loss"][-1], 4))
print("ON  val_loss:", round(hist_on["val_loss"][-1], 4),
      "| val_policy_loss:", round(hist_on["val_policy_loss"][-1], 4))


OFF val_loss: 0.3774 | val_policy_loss: 2.4096
ON  val_loss: 2.0039 | val_policy_loss: 1.6311


In [36]:
# Cell 5: fixed test set + eval helper (NN-only solve rate)
def make_test_cases(n=20, seed=77):
    rng = random.Random(seed)
    cases = []
    for _ in range(n):
        perm = list(range(len(SOMA_PIECES)))
        rng.shuffle(perm)
        cases.append([SOMA_PIECES[i] for i in perm])
    return cases

def eval_nn_model(model, cases, beam_width=32, timeout=8.0):
    solved = 0
    times = []
    for pieces in cases:
        t0 = time.time()
        sol = ns.nn_solve(
            pieces, 3, model,
            max_pieces=model.in_channels - 1,
            beam_width=beam_width,
            timeout=timeout,
            device="cpu",
        )
        dt = time.time() - t0
        times.append(dt)
        if sol is not None:
            solved += 1

    return {
        "solved": solved,
        "total": len(cases),
        "solve_rate": solved / len(cases),
        "avg_time": float(np.mean(times)),
        "p95_time": float(np.percentile(times, 95)),
    }

cases = make_test_cases(n=20, seed=77)
res_off = eval_nn_model(model_off, cases, beam_width=32, timeout=8.0)
res_on  = eval_nn_model(model_on,  cases, beam_width=32, timeout=8.0)

print("OFF:", res_off)
print("ON :", res_on)


OFF: {'solved': 20, 'total': 20, 'solve_rate': 1.0, 'avg_time': 2.1698696613311768, 'p95_time': 2.4677302241325383}
ON : {'solved': 20, 'total': 20, 'solve_rate': 1.0, 'avg_time': 1.9578383207321166, 'p95_time': 2.062138020992279}


In [37]:
# Cell 6: compact comparison print
delta = res_on["solve_rate"] - res_off["solve_rate"]
print("solve_rate delta (ON - OFF):", round(delta, 3))
print("avg_time OFF/ON:", round(res_off["avg_time"], 3), "/", round(res_on["avg_time"], 3))


solve_rate delta (ON - OFF): 0.0
avg_time OFF/ON: 2.17 / 1.958


In [39]:
# harder test cases, looks like policy lambda = 1.0 is not good
cases = make_test_cases(n=60, seed=101)
res_off_hard = eval_nn_model(model_off, cases, beam_width=8, timeout=3.0)
res_on_hard  = eval_nn_model(model_on,  cases, beam_width=8, timeout=3.0)
print("OFF hard:", res_off_hard)
print("ON  hard:", res_on_hard)


OFF hard: {'solved': 59, 'total': 60, 'solve_rate': 0.9833333333333333, 'avg_time': 0.6897371967633565, 'p95_time': 0.976926779747009}
ON  hard: {'solved': 52, 'total': 60, 'solve_rate': 0.8666666666666667, 'avg_time': 0.6438149293263753, 'p95_time': 0.7320869803428642}


In [40]:
# Sweep policy-loss weights (lambda_policy) to find the best solve-rate/runtime tradeoff on a harder test set
lambdas = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]
cases_hard = make_test_cases(n=60, seed=101)

rows = []
for lp in lambdas:
    m, h = train_variant(lambda_policy=lp, epochs=6, seed=2026)
    r = eval_nn_model(m, cases_hard, beam_width=8, timeout=3.0)
    rows.append({
        "lambda_policy": lp,
        "solve_rate": r["solve_rate"],
        "avg_time": r["avg_time"],
        "p95_time": r["p95_time"],
        "val_value_loss": h["val_value_loss"][-1],
        "val_policy_loss": h["val_policy_loss"][-1],
    })

rows = sorted(rows, key=lambda x: (-x["solve_rate"], x["avg_time"]))
for row in rows:
    print(row)


{'lambda_policy': 0.0, 'solve_rate': 0.9833333333333333, 'avg_time': 0.623330827554067, 'p95_time': 0.722218608856201, 'val_value_loss': 0.37744906213548446, 'val_policy_loss': 2.4095911847220526}
{'lambda_policy': 0.3, 'solve_rate': 0.95, 'avg_time': 0.5729765097300211, 'p95_time': 0.5963934779167175, 'val_value_loss': 0.3690985441207886, 'val_policy_loss': 1.8006908761130438}
{'lambda_policy': 0.5, 'solve_rate': 0.9333333333333333, 'avg_time': 0.5726686954498291, 'p95_time': 0.6031498074531555, 'val_value_loss': 0.36040272646480137, 'val_policy_loss': 1.7133578724331326}
{'lambda_policy': 0.1, 'solve_rate': 0.9, 'avg_time': 0.5668970823287964, 'p95_time': 0.6084433436393738, 'val_value_loss': 0.3760683271620009, 'val_policy_loss': 2.148473170068529}
{'lambda_policy': 0.2, 'solve_rate': 0.8833333333333333, 'avg_time': 0.5682407816251119, 'p95_time': 0.5963978052139283, 'val_value_loss': 0.35052944554222953, 'val_policy_loss': 1.8215412166383531}
{'lambda_policy': 1.0, 'solve_rate': 0.

In [41]:
# Summary of lambda_policy sweep: pick robust default for current project settings
best = max(rows, key=lambda r: r["solve_rate"])
print("Best by solve_rate:", best)
print("Chosen default lambda_policy:", 0.0)


Best by solve_rate: {'lambda_policy': 0.0, 'solve_rate': 0.9833333333333333, 'avg_time': 0.623330827554067, 'p95_time': 0.722218608856201, 'val_value_loss': 0.37744906213548446, 'val_policy_loss': 2.4095911847220526}
Chosen default lambda_policy: 0.0


In [42]:
# Final recommendation note for report/notebook
print(
    "Policy-action redesign is implemented, but under current data/model regime "
    "policy loss > 0 reduces hard-case solve rate. "
    "Use lambda_policy=0.0 as default; revisit after more data or longer training."
)


Policy-action redesign is implemented, but under current data/model regime policy loss > 0 reduces hard-case solve rate. Use lambda_policy=0.0 as default; revisit after more data or longer training.


In [43]:
# Project status and remaining TODOs for report/notebook
print("""
## Project Status (as of now)

### Completed Changes
1. Policy-guided beam pruning replaced random truncation
   - Search now ranks placements with model policy logits (deterministic top-K).
2. Beam-level policy inference batching
   - Policy logits computed once per beam depth (less repeated model calls).
3. ADI label-noise reduction
   - Failed-search states are no longer blindly labeled negative by default.
   - Optional DLX verification used for higher-confidence negatives.
4. DLX timeout guard in hybrid solver
   - Fallback DLX runs with enforced timeout to prevent stalls.
5. Leakage-safe data split
   - Train/val split is now grouped by instance_id (no instance overlap).
6. Policy target redesign (major)
   - Policy training target moved from voxel argmax proxy to placement-action candidates.

### Current Empirical Conclusion
- On easy settings, policy loss ON can improve runtime.
- On harder settings, policy loss ON reduced solve rate.
- Best hard-case result so far: lambda_policy = 0.0 (value-only objective).

### Current Recommended Defaults
- Keep policy-action redesign code in place.
- Use lambda_policy = 0.0 in config for robust solve rate today.
- Keep policy-guided inference/pruning active (already beneficial structurally).

## Remaining TODOs

### High Priority
1. Tune policy contribution instead of full ON/OFF
   - Try small lambda_policy values with longer training (e.g., 0.02, 0.05, 0.1).
   - Consider curriculum: start at 0.0, warm up to >0 after value stabilizes.
2. Improve policy data quality/quantity
   - More instances, stronger negatives, and broader puzzle diversity.
3. Add fixed benchmark harness script/notebook block
   - Report solve rate + avg/p95 time for NN-only, DLX-only, and hybrid.

### Medium Priority
4. Add deterministic experiment controls
   - Centralized seed setting and logging in training/eval runs.
5. Add memoization for repeated geometry ops
   - Cache orientations/placements for repeated pieces to reduce overhead.

### Report/Presentation TODOs
6. Document ablation findings clearly
   - Show why lambda_policy currently stays 0.0.
7. Include one “NN fails, DLX fallback succeeds” example
   - Demonstrates practical value of hybrid architecture.
8. Include short reproducibility recipe
   - Environment, commands, and exact benchmark settings.
""")



## Project Status (as of now)

### Completed Changes
1. Policy-guided beam pruning replaced random truncation
   - Search now ranks placements with model policy logits (deterministic top-K).
2. Beam-level policy inference batching
   - Policy logits computed once per beam depth (less repeated model calls).
3. ADI label-noise reduction
   - Failed-search states are no longer blindly labeled negative by default.
   - Optional DLX verification used for higher-confidence negatives.
4. DLX timeout guard in hybrid solver
   - Fallback DLX runs with enforced timeout to prevent stalls.
5. Leakage-safe data split
   - Train/val split is now grouped by instance_id (no instance overlap).
6. Policy target redesign (major)
   - Policy training target moved from voxel argmax proxy to placement-action candidates.

### Current Empirical Conclusion
- On easy settings, policy loss ON can improve runtime.
- On harder settings, policy loss ON reduced solve rate.
- Best hard-case result so far: lambda_pol

In [44]:
## Proxy Grading Check (20-case, 15/20 bar)

In [45]:
# Load the grading harness and run a 20-case proxy grading pass for your hybrid solver
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import grading_harness as gh
importlib.reload(gh)

report = gh.run_proxy_grading(
    mode="hybrid",
    model_name="soma_3x3x3_quick",
    beam_width=32,
    timeout_nn=8.0,
    timeout_dlx=20.0,
    seed=561,
    n_cases=20,
)
gh.print_proxy_grading_report(report)


Target: 2x2x2 cube (8 cells, 8 pieces)
  Piece 0: 1 cubes, 8 placements
  Piece 1: 1 cubes, 8 placements
  Piece 2: 1 cubes, 8 placements
  Piece 3: 1 cubes, 8 placements
  Piece 4: 1 cubes, 8 placements
  Piece 5: 1 cubes, 8 placements
  Piece 6: 1 cubes, 8 placements
  Piece 7: 1 cubes, 8 placements
  DLX matrix: 64 rows x 16 columns
Solving...
Found 1 solution(s).
Target: 2x2x2 cube (8 cells, 4 pieces)
  Piece 0: 2 cubes, 12 placements
  Piece 1: 2 cubes, 12 placements
  Piece 2: 2 cubes, 12 placements
  Piece 3: 2 cubes, 12 placements
  DLX matrix: 48 rows x 12 columns
Solving...
Found 1 solution(s).
Target: 2x2x2 cube (8 cells, 6 pieces)
  Piece 0: 3 cubes, 24 placements
  Piece 1: 1 cubes, 8 placements
  Piece 2: 1 cubes, 8 placements
  Piece 3: 1 cubes, 8 placements
  Piece 4: 1 cubes, 8 placements
  Piece 5: 1 cubes, 8 placements
  DLX matrix: 64 rows x 14 columns
Solving...
Found 1 solution(s).
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 27 placements
  Piece 1

In [48]:
# Print only the key grade-like summary
score = report["score"]
print("correct:", score["correct"], "/", score["total"])
print("passes_a_plus_bar:", score["passes_a_plus_bar"])
print("accuracy:", round(score["accuracy"], 3))
print("avg_time_sec:", round(score["avg_time_sec"], 3))
print("p95_time_sec:", round(score["p95_time_sec"], 3))


correct: 20 / 20
passes_a_plus_bar: True
accuracy: 1.0
avg_time_sec: 1.447
p95_time_sec: 2.209


In [49]:
# Show only failed cases to focus debugging effort
fails = [r for r in report["results"] if not r["correct"]]
print("num_failures:", len(fails))
for r in fails:
    print(r["case_id"], "| expected:", r["expected_solvable"], "| predicted:", r["predicted_solvable"], "| reason:", r["reason"])


num_failures: 0


In [51]:
## Stress Check: Multi-Seed + Solvable-Only + No-Easy-Cases

In [52]:
# Reload updated grading harness
import sys, importlib
sys.path.insert(0, "/Users/nick/Desktop/Polycube Solver")

import grading_harness as gh
importlib.reload(gh)
print("reloaded")


reloaded


In [53]:
# Single-seed harder proxy run (no easy fixed cases)
hard_report = gh.run_proxy_grading(
    mode="hybrid",
    model_name="soma_3x3x3_quick",
    beam_width=32,
    timeout_nn=8.0,
    timeout_dlx=20.0,
    seed=561,
    n_cases=20,
    include_fixed_cases=False,
)
gh.print_proxy_grading_report(hard_report)

Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 27 placements
  Piece 1: 4 cubes, 72 placements
  Piece 2: 3 cubes, 27 placements
Piece 3 ([(0, 1, 0), (1, 2, 0), (0, 2, 0), (0, 0, 0), (1, 3, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 3 cubes, 144 placements
  Piece 1: 4 cubes, 36 placements
  Piece 2: 5 cubes, 192 placements
  Piece 3: 4 cubes, 64 placements
  Piece 4: 3 cubes, 144 placements
  Piece 5: 3 cubes, 144 placements
Piece 6 ([(0, 1, 0), (0, 2, 0), (0, 0, 0), (0, 3, 0), (0, 4, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 5 cubes, 48 placements
  Piece 1: 3 cubes, 27 placements
  Piece 2: 5 cubes, 96 placements
  Piece 3: 4 cubes, 36 placements
  Piece 4: 3 cubes, 27 placements
  Piece 5: 3 cubes, 144 placements
  Piece 6: 4 cubes, 72 placements
  DLX matrix: 450 rows x 34 columns
Solving...
Found 1 solution(s).
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 5 cubes, 96 placements
  Piece

In [54]:
# Multi-seed harder proxy summary
multi = gh.run_multi_seed_proxy_grading(
    seeds=[101, 202, 303, 404, 505],
    mode="hybrid",
    model_name="soma_3x3x3_quick",
    beam_width=32,
    timeout_nn=8.0,
    timeout_dlx=20.0,
    n_cases=20,
    include_fixed_cases=False,
)
gh.print_multi_seed_report(multi)


Target: 3x3x3 cube (27 cells, 6 pieces)
  Piece 0: 5 cubes, 36 placements
  Piece 1: 5 cubes, 192 placements
  Piece 2: 5 cubes, 36 placements
  Piece 3: 4 cubes, 36 placements
Piece 4 ([(1, 1, 0), (0, 1, 0), (2, 1, 0), (3, 1, 0), (1, 0, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 7 pieces)
Piece 0 ([(0, 2, 0), (0, 0, 0), (0, 3, 0), (0, 1, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 5 cubes, 36 placements
Piece 1 ([(1, 1, 0), (0, 1, 0), (2, 1, 0), (0, 0, 0), (3, 1, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 6 pieces)
Piece 0 ([(0, 1, 0), (1, 2, 0), (0, 2, 0), (0, 0, 0), (1, 3, 0)]) has no valid placements!
Target: 3x3x3 cube (27 cells, 7 pieces)
  Piece 0: 5 cubes, 192 placements
  Piece 1: 4 cubes, 72 placements
  Piece 2: 3 cubes, 144 placements
  Piece 3: 4 cubes, 72 placements
  Piece 4: 3 cubes, 27 placements
Piece 5 ([(0, 2, 0), (0, 0, 0), (0, 3, 0), (0, 1, 0)]) has no valid placements!
Target: 3x3x3 cube (27 c

In [56]:
# Optional: NN-only stress (no DLX fallback) to reveal heuristic strength
multi_nn = gh.run_multi_seed_proxy_grading(
    seeds=[101, 202, 303],
    mode="nn",
    model_name="soma_3x3x3_quick",
    beam_width=32,
    timeout_nn=8.0,
    n_cases=20,
    include_fixed_cases=False,
)
gh.print_multi_seed_report(multi_nn)


Multi-Seed Proxy Grading Summary
Seeds: 3  Mean acc: 0.800  Min/Max acc: 0.750/0.900
Aggregate correct: 48/60 (0.800)
Mean solvable acc: 0.588  Mean unsolvable acc: 1.000
------------------------------------------------------------------------
seed=101  correct=18/20  acc=0.900  solv=0.818  unsolv=1.000
seed=202  correct=15/20  acc=0.750  solv=0.500  unsolv=1.000
seed=303  correct=15/20  acc=0.750  solv=0.444  unsolv=1.000
